# Dynamic-DeepHit for Competing Risks

This notebook implements **Dynamic-DeepHit** (Lee et al., 2020) for competing risks analysis using the **full longitudinal loan-month panel data**. Unlike the static DeepHit (notebook 08) which uses only terminal observations, Dynamic-DeepHit processes the complete monthly history via a GRU + temporal attention mechanism, enabling dynamic predictions that update as new data arrives.

## Methodology

| Aspect | Static DeepHit (NB 08) | Dynamic-DeepHit (this NB) |
|--------|----------------------|---------------------------|
| **Input** | Terminal observation (1 per loan) | Full monthly sequence |
| **Backbone** | Shared FFN | GRU + Temporal Attention |
| **Prediction** | Static (one-shot) | Dynamic (updates monthly) |
| **Loss** | L1 (NLL) + L2 (Ranking) | L1 + L2 + L3 (Next-step) |
| **Parameters** | ~722K | ~230K |

## Architecture

```
Input: x_padded (batch, seq_len, 21), lengths (batch,)
  ├─ Input Embedding: Linear(21→64) + ReLU
  ├─ GRU (2 layers, hidden=128, dropout=0.2)
  │   └─ h_all: (batch, seq_len, 128)
  ├─ Temporal Attention
  │   ├─ Score: f_a([h_j; x_J]) → scalar per timestep
  │   ├─ Masked softmax → attention weights a_j
  │   └─ Context: c = Σ(a_j · h_j) → (batch, 128)
  ├─ Concatenate: [c; x_J] → (batch, 149)
  ├─ Cause-Specific Heads
  │   ├─ Prepay MLP: 149→128→64→num_time_bins
  │   └─ Default MLP: 149→128→64→num_time_bins
  ├─ Joint Softmax → PMF (batch, 2, num_time_bins)
  └─ Next-Step Predictor: Linear(128→16) on h_all → x_pred (for L3)
```

## Loss Function

$$\mathcal{L} = \mathcal{L}_1 + \alpha \cdot \mathcal{L}_2 + \beta \cdot \mathcal{L}_3$$

- **L1**: Conditional NLL (default events weighted 50x for class imbalance)
- **L2**: Cause-specific ranking loss (α_prepay=0.2, α_default=1.0)
- **L3**: Next-step prediction MSE for time-varying covariates (β=0.1)

## References

- Lee, C., Yoon, J., & van der Schaar, M. (2020). Dynamic-DeepHit: A Deep Learning Approach for Dynamic Survival Analysis with Competing Risks Based on Longitudinal Data. IEEE TBME.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import pickle
import warnings
warnings.filterwarnings('ignore')

# Deep learning
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

# Survival analysis metrics
from sksurv.util import Surv
from sksurv.metrics import concordance_index_censored, concordance_index_ipcw

# Project modules
import sys
sys.path.insert(0, '..')
from src.competing_risks.dynamic_deephit import (
    DynamicDeepHitNetwork,
    DynamicDeepHitLoss,
    MortgageSequenceDataset,
    collate_mortgage_sequences,
    preprocess_panel_to_sequences,
    ALL_FEATURES,
    STATIC_FEATURES,
    BEHAVIORAL_FEATURES,
    MACRO_FEATURES,
    TIME_VARYING_FEATURES,
)

sns.set_style('whitegrid')
%matplotlib inline

# Seeds
np.random.seed(42)
torch.manual_seed(42)

# Device
if torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
    print('Using MPS (Apple Silicon GPU)')
elif torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    print(f'Using CUDA ({torch.cuda.get_device_name(0)})')
else:
    DEVICE = torch.device('cpu')
    print('Using CPU')

TIME_HORIZONS = [24, 48, 72]
print(f'PyTorch version: {torch.__version__}')
print(f'Device: {DEVICE}')

In [ ]:
# === CONFIGURATION ===
DATA_DIR = Path('../data/processed')
FIGURES_DIR = Path('../reports/figures')
MODELS_DIR = Path('../models')

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Cross-validation folds (Blumenstock methodology)
TRAIN_FOLDS = list(range(10))
VAL_FOLDS = [9]
TEST_FOLD = 10

# Dynamic-DeepHit hyperparameters
MAX_SEQ_LEN = 120
NUM_TIME_BINS = 120

DDEEPHIT_PARAMS = {
    # Architecture
    'embed_dim': 64,
    'hidden_dim': 128,
    'num_rnn_layers': 2,
    'head_hidden1': 128,
    'head_hidden2': 64,
    'dropout': 0.2,
    # Training
    'batch_size': 64,
    'epochs': 100,
    'learning_rate': 0.001,
    'grad_accum_steps': 4,
    # Loss
    'alpha_prepay': 0.2,
    'alpha_default': 1.0,
    'sigma': 0.1,
    'beta': 0.1,
    'default_event_weight': 50.0,
}

# Dynamic evaluation
LANDMARK_TIMES = [12, 24, 36, 48, 60]
PREDICTION_HORIZONS = [12, 24, 36]

EVENT_NAMES = {0: 'Censored', 1: 'Prepay', 2: 'Default'}

print(f'Max sequence length: {MAX_SEQ_LEN}')
print(f'Time bins: {NUM_TIME_BINS}')
print(f'\nDynamic-DeepHit parameters:')
for k, v in DDEEPHIT_PARAMS.items():
    print(f'  {k}: {v}')

---

## Load Panel Data + Sequence Statistics

In [ ]:
print('Loading loan-month panel data...')
panel_df = pd.read_parquet(DATA_DIR / 'loan_month_panel.parquet')

print(f'Loaded {len(panel_df):,} loan-months')
print(f'Unique loans: {panel_df["loan_sequence_number"].nunique():,}')
print(f'Folds: {sorted(panel_df["fold"].unique())}')

# Sequence length distribution
seq_lengths = panel_df.groupby('loan_sequence_number').size()
print(f'\nSequence lengths:')
print(f'  Mean: {seq_lengths.mean():.1f}')
print(f'  Median: {seq_lengths.median():.0f}')
print(f'  Min: {seq_lengths.min()}, Max: {seq_lengths.max()}')
print(f'  P85: {seq_lengths.quantile(0.85):.0f} (max_seq_len={MAX_SEQ_LEN})')

# Event distribution
print('\nEvent distribution (terminal observations):')
terminal_events = panel_df[panel_df['event'] == 1].groupby('event_code').size()
for code, count in terminal_events.items():
    print(f'  {EVENT_NAMES.get(code, "Other")} (k={code}): {count:,}')

# Feature availability
feature_cols = [f for f in ALL_FEATURES if f in panel_df.columns]
print(f'\nAvailable features: {len(feature_cols)}/21')
print(f'  Static: {sum(1 for f in STATIC_FEATURES if f in panel_df.columns)}/5')
print(f'  Behavioral: {sum(1 for f in BEHAVIORAL_FEATURES if f in panel_df.columns)}/4')
print(f'  Macro: {sum(1 for f in MACRO_FEATURES if f in panel_df.columns)}/12')

---

## Feature Engineering + Sequence Dataset Creation

In [ ]:
print('=== Preprocessing Sequences ===')

train_folds_actual = [f for f in TRAIN_FOLDS if f not in VAL_FOLDS]

# Split panel by fold
train_panel = panel_df[panel_df['fold'].isin(train_folds_actual)].copy()
val_panel = panel_df[panel_df['fold'].isin(VAL_FOLDS)].copy()
test_panel = panel_df[panel_df['fold'] == TEST_FOLD].copy()

print(f'Train panel: {len(train_panel):,} loan-months')
print(f'Val panel: {len(val_panel):,} loan-months')
print(f'Test panel: {len(test_panel):,} loan-months')

# Preprocess train (fits scaler)
train_seqs, scaler, final_feature_cols = preprocess_panel_to_sequences(
    train_panel, feature_cols, max_seq_len=MAX_SEQ_LEN, fit_scaler=True)
print(f'\nTrain sequences: {len(train_seqs):,}')

# Preprocess val/test (uses fitted scaler)
val_seqs, _, _ = preprocess_panel_to_sequences(
    val_panel, feature_cols, max_seq_len=MAX_SEQ_LEN, scaler=scaler)
test_seqs, _, _ = preprocess_panel_to_sequences(
    test_panel, feature_cols, max_seq_len=MAX_SEQ_LEN, scaler=scaler)
print(f'Val sequences: {len(val_seqs):,}')
print(f'Test sequences: {len(test_seqs):,}')

# Sequence length stats
train_lengths = [s['length'] for s in train_seqs]
print(f'\nSequence lengths (train, after truncation):')
print(f'  Mean: {np.mean(train_lengths):.1f}, Median: {np.median(train_lengths):.0f}')
print(f'  Min: {np.min(train_lengths)}, Max: {np.max(train_lengths)}')

# Event distribution
print('\nEvent distribution (train sequences):')
train_events = [s['event'] for s in train_seqs]
for code in sorted(set(train_events)):
    count = sum(1 for e in train_events if e == code)
    print(f'  {EVENT_NAMES.get(code, "Other")} (k={code}): {count:,}')

print(f'\nFinal features ({len(final_feature_cols)}): {final_feature_cols}')

In [ ]:
# Create PyTorch datasets and data loaders
print('=== Creating Datasets ===')

train_dataset = MortgageSequenceDataset(train_seqs, max_seq_len=MAX_SEQ_LEN)
val_dataset = MortgageSequenceDataset(val_seqs, max_seq_len=MAX_SEQ_LEN)
test_dataset = MortgageSequenceDataset(test_seqs, max_seq_len=MAX_SEQ_LEN)

BS = DDEEPHIT_PARAMS['batch_size']
train_loader = DataLoader(
    train_dataset, batch_size=BS, shuffle=True,
    collate_fn=collate_mortgage_sequences)
val_loader = DataLoader(
    val_dataset, batch_size=BS * 2, shuffle=False,
    collate_fn=collate_mortgage_sequences)
test_loader = DataLoader(
    test_dataset, batch_size=BS * 2, shuffle=False,
    collate_fn=collate_mortgage_sequences)

# Time bins
all_durations = np.array([s['duration'] for s in train_seqs + val_seqs + test_seqs])
time_bins = np.linspace(all_durations.min(), all_durations.max(), NUM_TIME_BINS + 1)
time_bins_tensor = torch.tensor(time_bins, dtype=torch.float32)
time_points = (time_bins[:-1] + time_bins[1:]) / 2

print(f'Train batches: {len(train_loader)}')
print(f'Val batches: {len(val_loader)}')
print(f'Test batches: {len(test_loader)}')
print(f'Time bins: {NUM_TIME_BINS}, width={np.diff(time_bins).mean():.2f} months')

# Compute TV feature indices
tv_feature_indices = [final_feature_cols.index(f) for f in TIME_VARYING_FEATURES
                      if f in final_feature_cols]
print(f'Time-varying features for L3: {len(tv_feature_indices)}')

# Verify a batch
x_b, l_b, d_b, e_b = next(iter(train_loader))
print(f'\nSample batch: x={x_b.shape}, lengths={l_b.shape}, '
      f'durations={d_b.shape}, events={e_b.shape}')

---

## Model Architecture Summary

In [ ]:
# Build model
in_features = len(final_feature_cols)

model = DynamicDeepHitNetwork(
    in_features=in_features,
    num_time_bins=NUM_TIME_BINS,
    num_causes=2,
    embed_dim=DDEEPHIT_PARAMS['embed_dim'],
    hidden_dim=DDEEPHIT_PARAMS['hidden_dim'],
    num_rnn_layers=DDEEPHIT_PARAMS['num_rnn_layers'],
    head_hidden1=DDEEPHIT_PARAMS['head_hidden1'],
    head_hidden2=DDEEPHIT_PARAMS['head_hidden2'],
    dropout=DDEEPHIT_PARAMS['dropout'],
    num_tv_features=len(tv_feature_indices),
)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print('Dynamic-DeepHit Architecture')
print('=' * 60)
print(f'Input Embedding: {in_features} -> {DDEEPHIT_PARAMS["embed_dim"]}')
print(f'GRU: {DDEEPHIT_PARAMS["num_rnn_layers"]} layers x {DDEEPHIT_PARAMS["hidden_dim"]} hidden')
print(f'Temporal Attention: score({DDEEPHIT_PARAMS["hidden_dim"]}+{in_features}) -> 1')
print(f'Cause heads: {DDEEPHIT_PARAMS["head_hidden1"]} -> {DDEEPHIT_PARAMS["head_hidden2"]} -> {NUM_TIME_BINS}')
print(f'Next-step predictor: {DDEEPHIT_PARAMS["hidden_dim"]} -> {len(tv_feature_indices)}')
print(f'Output: Joint softmax over {2 * NUM_TIME_BINS} (cause, time)')
print(f'\nTotal parameters: {n_params:,}')
print(f'(vs ~722K for static DeepHit)')

# Module breakdown
print('\nParameter breakdown:')
for name, module in model.named_children():
    n = sum(p.numel() for p in module.parameters() if p.requires_grad)
    print(f'  {name}: {n:,}')

---

## Training

In [ ]:
# Loss function
criterion = DynamicDeepHitLoss(
    alpha_prepay=DDEEPHIT_PARAMS['alpha_prepay'],
    alpha_default=DDEEPHIT_PARAMS['alpha_default'],
    sigma=DDEEPHIT_PARAMS['sigma'],
    beta=DDEEPHIT_PARAMS['beta'],
    default_event_weight=DDEEPHIT_PARAMS['default_event_weight'],
    num_tv_features=len(tv_feature_indices),
    tv_feature_indices=tv_feature_indices,
)

# Training function (inlined for notebook)
model = model.to(DEVICE)
time_bins_dev = time_bins_tensor.to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=DDEEPHIT_PARAMS['learning_rate'])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5)

grad_accum = DDEEPHIT_PARAMS['grad_accum_steps']
patience = 10

history = {k: [] for k in [
    'train_loss', 'val_loss', 'train_l1', 'train_l2', 'train_l3',
    'val_l1', 'val_l2', 'val_l3']}

best_val_loss = float('inf')
best_epoch = 0
best_state = None
epochs_no_improve = 0

print(f'Training on {DEVICE}')
print(f'{"Epoch":>6} | {"Train":>10} | {"Val":>10} | '
      f'{"L1":>8} | {"L2":>8} | {"L3":>8}')
print('-' * 65)

for epoch in range(DDEEPHIT_PARAMS['epochs']):
    model.train()
    train_m = {k: [] for k in ['loss', 'l1', 'l2', 'l3']}
    optimizer.zero_grad()

    for step, (x_pad, lens, durs, evts) in enumerate(train_loader):
        x_pad = x_pad.to(DEVICE)
        lens = lens.to(DEVICE)
        durs = durs.to(DEVICE)
        evts = evts.to(DEVICE)

        pmf, x_pred = model(x_pad, lens)
        loss, l1, l2, l3 = criterion(
            pmf, x_pred, x_pad, lens, durs, evts, time_bins_dev)
        (loss / grad_accum).backward()

        if (step + 1) % grad_accum == 0:
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad()

        train_m['loss'].append(loss.item())
        train_m['l1'].append(l1.item())
        train_m['l2'].append(l2.item())
        train_m['l3'].append(l3.item())

    # Flush remaining gradients
    if (step + 1) % grad_accum != 0:
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        optimizer.zero_grad()

    avg_train = {k: np.mean(v) for k, v in train_m.items()}

    # Validation
    model.eval()
    val_m = {k: [] for k in ['loss', 'l1', 'l2', 'l3']}
    with torch.no_grad():
        for x_pad, lens, durs, evts in val_loader:
            x_pad = x_pad.to(DEVICE)
            lens = lens.to(DEVICE)
            durs = durs.to(DEVICE)
            evts = evts.to(DEVICE)
            pmf, x_pred = model(x_pad, lens)
            loss, l1, l2, l3 = criterion(
                pmf, x_pred, x_pad, lens, durs, evts, time_bins_dev)
            val_m['loss'].append(loss.item())
            val_m['l1'].append(l1.item())
            val_m['l2'].append(l2.item())
            val_m['l3'].append(l3.item())

    avg_val = {k: np.mean(v) for k, v in val_m.items()}
    scheduler.step(avg_val['loss'])

    for k in ['loss', 'l1', 'l2', 'l3']:
        history[f'train_{k}' if k != 'loss' else 'train_loss'].append(avg_train[k])
        history[f'val_{k}' if k != 'loss' else 'val_loss'].append(avg_val[k])

    if avg_val['loss'] < best_val_loss:
        best_val_loss = avg_val['loss']
        best_epoch = epoch
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1

    if epoch % 5 == 0 or epochs_no_improve >= patience:
        print(f'{epoch:>6} | {avg_train["loss"]:>10.4f} | {avg_val["loss"]:>10.4f} | '
              f'{avg_val["l1"]:>8.4f} | {avg_val["l2"]:>8.4f} | {avg_val["l3"]:>8.4f}')

    if epochs_no_improve >= patience:
        print(f'\nEarly stopping at epoch {epoch}')
        break

if best_state is not None:
    model.load_state_dict(best_state)
    model.to(DEVICE)

print(f'\nBest epoch: {best_epoch} (val_loss={best_val_loss:.4f})')

---

## Training Curves (L1, L2, L3 Components)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4))

for ax, (train_key, val_key, title) in zip(axes, [
    ('train_loss', 'val_loss', 'Total Loss'),
    ('train_l1', 'val_l1', 'L1: NLL'),
    ('train_l2', 'val_l2', 'L2: Ranking'),
    ('train_l3', 'val_l3', 'L3: Next-Step'),
]):
    ax.plot(history[train_key], label='Train')
    ax.plot(history[val_key], label='Validation')
    ax.set_xlabel('Epoch')
    ax.set_ylabel(title)
    ax.set_title(f'Dynamic-DeepHit: {title}')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'dynamic_deephit_training_curves.png',
            dpi=150, bbox_inches='tight')
plt.show()

---

## Static Evaluation (C-index at 24/48/72 — Comparison with Static DeepHit)

In [ ]:
print('=== Static Evaluation ===')

# Predict on test set
model.eval()
all_cif, all_surv, all_attn = [], [], []

with torch.no_grad():
    for x_pad, lens, durs, evts in test_loader:
        x_pad = x_pad.to(DEVICE)
        lens = lens  # keep on CPU for pack_padded_sequence

        # Single forward pass → compute CIF and survival from PMF
        pmf, _ = model(x_pad, lens)
        cif = torch.cumsum(pmf, dim=-1)
        surv = 1.0 - cif.sum(dim=1)
        attn = model.get_attention_weights()

        all_cif.append(cif.cpu().numpy())
        all_surv.append(surv.cpu().numpy())
        if attn is not None:
            all_attn.append(attn.cpu().numpy())

cif_np = np.concatenate(all_cif, axis=0)
surv_np = np.concatenate(all_surv, axis=0)

# Attention weights may have different seq_len per batch due to padding;
# pad to a common max length before concatenation
if all_attn:
    max_t = max(a.shape[1] for a in all_attn)
    padded_attn = []
    for a in all_attn:
        if a.shape[1] < max_t:
            pad_width = max_t - a.shape[1]
            a = np.pad(a, ((0, 0), (0, pad_width)), constant_values=0.0)
        padded_attn.append(a)
    attn_np = np.concatenate(padded_attn, axis=0)
else:
    attn_np = None

cif_prepay = cif_np[:, 0, :]
cif_default = cif_np[:, 1, :]

print(f'CIF shapes: prepay={cif_prepay.shape}, default={cif_default.shape}')
print(f'Survival shape: {surv_np.shape}')
if attn_np is not None:
    print(f'Attention shape: {attn_np.shape}')

# Get test labels
duration_test = np.array([s['duration'] for s in test_seqs])
event_test = np.array([s['event'] for s in test_seqs])
duration_train = np.array([s['duration'] for s in train_seqs])
event_train = np.array([s['event'] for s in train_seqs])

def get_risk_at_horizon(cif, time_points, tau):
    idx = min(np.searchsorted(time_points, tau), len(time_points) - 1)
    return cif[:, idx]

# Evaluate both causes
for cause_name, cause_code, cif_cause in [
    ('PREPAYMENT', 1, cif_prepay), ('DEFAULT', 2, cif_default)
]:
    print(f'\n=== {cause_name} C-index ===')
    event_binary = (event_test == cause_code).astype(bool)
    y_train_sk = Surv.from_arrays(
        (event_train == cause_code).astype(bool), duration_train)
    y_test_sk = Surv.from_arrays(event_binary, duration_test)

    for tau in TIME_HORIZONS:
        try:
            risk = get_risk_at_horizon(cif_cause, time_points, tau)
            c_tau = concordance_index_ipcw(y_train_sk, y_test_sk, risk, tau=tau)
            print(f'  tau={tau:3d}: C-index (IPCW) = {c_tau[0]:.4f}')
        except Exception as e:
            print(f'  tau={tau:3d}: Error - {str(e)[:50]}')

    risk_overall = cif_cause.mean(axis=1)
    c_harrell = concordance_index_censored(event_binary, duration_test, risk_overall)[0]
    print(f'  Overall (Harrell): {c_harrell:.4f}')

In [ ]:
# Plot static C-index comparison
fig, ax = plt.subplots(figsize=(10, 6))

# Recompute for plotting
prepay_cindex, default_cindex = [], []
for cif_cause, cause_code in [(cif_prepay, 1), (cif_default, 2)]:
    event_binary = (event_test == cause_code).astype(bool)
    y_train_sk = Surv.from_arrays(
        (event_train == cause_code).astype(bool), duration_train)
    y_test_sk = Surv.from_arrays(event_binary, duration_test)
    vals = []
    for tau in TIME_HORIZONS:
        try:
            risk = get_risk_at_horizon(cif_cause, time_points, tau)
            c = concordance_index_ipcw(y_train_sk, y_test_sk, risk, tau=tau)[0]
            vals.append(c)
        except:
            vals.append(0)
    if cause_code == 1:
        prepay_cindex = vals
    else:
        default_cindex = vals

x = np.arange(len(TIME_HORIZONS))
width = 0.35
bars1 = ax.bar(x - width/2, prepay_cindex, width, label='Prepayment',
               color='steelblue', alpha=0.8)
bars2 = ax.bar(x + width/2, default_cindex, width, label='Default',
               color='indianred', alpha=0.8)

for bar, val in zip(bars1, prepay_cindex):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', va='bottom', fontsize=10)
for bar, val in zip(bars2, default_cindex):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', va='bottom', fontsize=10)

ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Random')
ax.set_xlabel('Time Horizon (months)')
ax.set_ylabel('C-index (IPCW)')
ax.set_title('Dynamic-DeepHit: Time-Dependent Concordance Index')
ax.set_xticks(x)
ax.set_xticklabels([f'tau = {h}' for h in TIME_HORIZONS])
ax.set_ylim(0.4, 1.0)
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'dynamic_deephit_time_dependent_cindex.png',
            dpi=150, bbox_inches='tight')
plt.show()

---

## Dynamic Evaluation (Landmark Analysis + Heatmap)

Evaluate how predictions improve as more data becomes available. For each landmark age $t_L$, we truncate loan histories to $t_L$ months, predict from there, and evaluate C-index for events within the prediction horizon.

In [ ]:
print('=== Dynamic Evaluation (Landmark Analysis) ===')

time_col = 'loan_age'
event_col = 'event_code'
loan_id_col = 'loan_sequence_number'

# Apply same feature engineering to test panel for dynamic eval
test_panel_fe = panel_df[panel_df['fold'] == TEST_FOLD].copy()
if 'bal_repaid' in test_panel_fe.columns:
    test_panel_fe['bal_repaid'] = (
        test_panel_fe.groupby(loan_id_col)['bal_repaid']
        .shift(1).fillna(0.0)
    )
if 'orig_upb' in test_panel_fe.columns:
    test_panel_fe['log_upb'] = np.log(test_panel_fe['orig_upb'].clip(lower=1.0))

dynamic_results = {}

for t_L in LANDMARK_TIMES:
    dynamic_results[t_L] = {}

    # Find loans that survived past t_L
    loan_durations = test_panel_fe.groupby(loan_id_col)[time_col].max()
    eligible_loans = loan_durations[loan_durations > t_L].index

    if len(eligible_loans) < 50:
        print(f't_L={t_L}: too few eligible loans ({len(eligible_loans)})')
        continue

    # Build truncated sequences
    sequences, res_durations, res_events = [], [], []
    for loan_id in eligible_loans:
        loan_data = test_panel_fe[test_panel_fe[loan_id_col] == loan_id].sort_values(time_col)
        truncated = loan_data[loan_data[time_col] <= t_L]
        if len(truncated) == 0:
            continue
        x = truncated[final_feature_cols].values.astype('float32')
        if scaler is not None:
            x = scaler.transform(x).astype('float32')
        if len(x) > MAX_SEQ_LEN:
            x = x[-MAX_SEQ_LEN:]
        sequences.append(torch.tensor(x, dtype=torch.float32))
        res_durations.append(float(loan_data[time_col].iloc[-1]) - t_L)
        res_events.append(int(loan_data[event_col].iloc[-1]))

    if len(sequences) < 50:
        continue

    # Pad and predict
    x_pad = torch.nn.utils.rnn.pad_sequence(sequences, batch_first=True).to(DEVICE)
    seq_lens = torch.tensor([s.shape[0] for s in sequences], dtype=torch.long).to(DEVICE)

    model.eval()
    with torch.no_grad():
        cif_lm = model.predict_cif(x_pad, seq_lens).cpu().numpy()

    res_dur = np.array(res_durations)
    res_ev = np.array(res_events)

    for h in PREDICTION_HORIZONS:
        for cause_name, cause_code, cause_idx in [
            ('Prepay', 1, 0), ('Default', 2, 1)
        ]:
            key = f'{cause_name}_{h}'
            event_binary = (res_ev == cause_code).astype(bool)
            events_in_window = ((res_dur <= h) & (res_ev == cause_code)).sum()
            if events_in_window < 3:
                dynamic_results[t_L][key] = np.nan
                continue
            try:
                risk = get_risk_at_horizon(cif_lm[:, cause_idx, :], time_points, h)
                c_idx = concordance_index_censored(
                    event_binary & (res_dur <= h),
                    np.minimum(res_dur, h), risk)[0]
                dynamic_results[t_L][key] = c_idx
            except:
                dynamic_results[t_L][key] = np.nan

    prepay_v = [dynamic_results[t_L].get(f'Prepay_{h}', np.nan) for h in PREDICTION_HORIZONS]
    default_v = [dynamic_results[t_L].get(f'Default_{h}', np.nan) for h in PREDICTION_HORIZONS]
    print(f't_L={t_L:3d}: Prepay=[{", ".join(f"{v:.3f}" if not np.isnan(v) else "NaN" for v in prepay_v)}] '
          f'Default=[{", ".join(f"{v:.3f}" if not np.isnan(v) else "NaN" for v in default_v)}]')

In [ ]:
# Dynamic C-index heatmap
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, cause_name, cmap in zip(axes, ['Prepay', 'Default'], ['Blues', 'Reds']):
    matrix = np.full((len(LANDMARK_TIMES), len(PREDICTION_HORIZONS)), np.nan)
    for i, t_L in enumerate(LANDMARK_TIMES):
        for j, h in enumerate(PREDICTION_HORIZONS):
            key = f'{cause_name}_{h}'
            if t_L in dynamic_results and key in dynamic_results[t_L]:
                matrix[i, j] = dynamic_results[t_L][key]

    im = ax.imshow(matrix, cmap=cmap, aspect='auto', vmin=0.5, vmax=1.0)
    ax.set_xticks(range(len(PREDICTION_HORIZONS)))
    ax.set_xticklabels([f'{h}m' for h in PREDICTION_HORIZONS])
    ax.set_yticks(range(len(LANDMARK_TIMES)))
    ax.set_yticklabels([f'{t}m' for t in LANDMARK_TIMES])
    ax.set_xlabel('Prediction Horizon')
    ax.set_ylabel('Landmark Age')
    ax.set_title(f'{cause_name}: Dynamic C-index')

    for i in range(len(LANDMARK_TIMES)):
        for j in range(len(PREDICTION_HORIZONS)):
            val = matrix[i, j]
            if not np.isnan(val):
                ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                        fontsize=9, color='white' if val > 0.75 else 'black')

    plt.colorbar(im, ax=ax, shrink=0.8)

plt.suptitle('Dynamic-DeepHit: Dynamic C-index (Landmark x Horizon)', fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'dynamic_deephit_dynamic_cindex_heatmap.png',
            dpi=150, bbox_inches='tight')
plt.show()

---

## Attention Visualization (Which Months Matter Most)

In [ ]:
if attn_np is not None:
    lengths_test = np.array([s['length'] for s in test_seqs])

    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    axes = axes.flatten()

    np.random.seed(42)
    valid_idx = np.where(lengths_test > 10)[0]
    sample_idx = np.random.choice(valid_idx, size=min(6, len(valid_idx)), replace=False)

    for ax, idx in zip(axes, sample_idx):
        length = int(lengths_test[idx])
        weights = attn_np[idx, :length]
        ax.bar(range(length), weights, color='steelblue', alpha=0.7)
        ax.set_xlabel('Month')
        ax.set_ylabel('Attention Weight')
        ax.set_title(f'Loan {idx} (len={length})')
        ax.grid(True, alpha=0.3)

    plt.suptitle('Dynamic-DeepHit: Temporal Attention Weights', fontsize=13)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'dynamic_deephit_attention.png',
                dpi=150, bbox_inches='tight')
    plt.show()

    # Aggregate attention pattern
    print('\nAggregate attention pattern (relative position):')
    n_bins = 10
    relative_weights = np.zeros(n_bins)
    counts = np.zeros(n_bins)
    for i in range(len(attn_np)):
        length = int(lengths_test[i])
        if length < 2:
            continue
        w = attn_np[i, :length]
        positions = np.linspace(0, 1, length)
        bin_idx = np.clip((positions * n_bins).astype(int), 0, n_bins - 1)
        for j in range(length):
            relative_weights[bin_idx[j]] += w[j]
            counts[bin_idx[j]] += 1
    avg_weights = relative_weights / np.maximum(counts, 1)
    for b in range(n_bins):
        label = f'{b*10}-{(b+1)*10}%'
        bar = '#' * int(avg_weights[b] * 500)
        print(f'  {label:>8}: {avg_weights[b]:.4f} {bar}')
else:
    print('No attention weights available.')

---

## Dynamic Survival Curves (Evolving Predictions for Sample Loans)

In [ ]:
# Plot CIF and survival curves for sample loans
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
np.random.seed(42)
sample_idx = np.random.choice(len(surv_np), size=5, replace=False)

for ax, data, title, ylabel in zip(axes,
    [cif_prepay[sample_idx], cif_default[sample_idx], surv_np[sample_idx]],
    ['Prepayment CIF', 'Default CIF', 'Overall Survival S(t)'],
    ['Cumulative Incidence', 'Cumulative Incidence', 'Survival Probability'],
):
    for i, idx in enumerate(sample_idx):
        ax.plot(time_points, data[i], label=f'Loan {idx}', alpha=0.7)
    ax.set_xlabel('Time (months)')
    ax.set_ylabel(ylabel)
    ax.set_title(f'Dynamic-DeepHit: {title}')
    ax.legend(loc='lower right' if 'CIF' in title else 'lower left', fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'dynamic_deephit_survival_curves.png',
            dpi=150, bbox_inches='tight')
plt.show()

# Stacked CIF plot
fig, ax = plt.subplots(figsize=(10, 6))
mean_cif_prepay = cif_prepay.mean(axis=0)
mean_cif_default = cif_default.mean(axis=0)

ax.fill_between(time_points, 0, mean_cif_prepay,
                alpha=0.7, label='Prepay', color='steelblue')
ax.fill_between(time_points, mean_cif_prepay,
                mean_cif_prepay + mean_cif_default,
                alpha=0.7, label='Default', color='indianred')
ax.fill_between(time_points, mean_cif_prepay + mean_cif_default, 1,
                alpha=0.3, label='Survival', color='gray')

ax.set_xlabel('Time (months)')
ax.set_ylabel('Probability')
ax.set_title('Dynamic-DeepHit: Average Stacked CIF (Test Set)')
ax.legend(loc='center right')
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

## Feature Importance (Permutation-Based)

In [ ]:
print('=== Feature Importance (Permutation) ===')

def permutation_importance_dynamic(model, test_seqs, feature_cols, cause_idx,
                                   device, time_points, max_seq_len,
                                   n_repeats=3, batch_size=128):
    """Permutation importance for Dynamic-DeepHit."""
    model.eval()
    event_code = cause_idx + 1
    events = np.array([s['event'] for s in test_seqs])
    durations = np.array([s['duration'] for s in test_seqs])
    event_binary = (events == event_code).astype(bool)

    # Baseline prediction
    dataset = MortgageSequenceDataset(test_seqs, max_seq_len=max_seq_len)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False,
                        collate_fn=collate_mortgage_sequences)
    baseline_cif = []
    with torch.no_grad():
        for x_pad, lens, _, _ in loader:
            cif = model.predict_cif(x_pad.to(device), lens.to(device))
            baseline_cif.append(cif[:, cause_idx, :].cpu().numpy())
    baseline_cif = np.concatenate(baseline_cif)
    baseline_risk = baseline_cif.mean(axis=1)
    baseline_c = concordance_index_censored(event_binary, durations, baseline_risk)[0]

    importances, importances_std = [], []
    for feat_idx, feat_name in enumerate(feature_cols):
        scores = []
        for _ in range(n_repeats):
            # Create permuted sequences
            perm_seqs = []
            for s in test_seqs:
                x_perm = s['x'].copy()
                np.random.shuffle(x_perm[:, feat_idx])
                perm_seqs.append({**s, 'x': x_perm})

            perm_dataset = MortgageSequenceDataset(perm_seqs, max_seq_len=max_seq_len)
            perm_loader = DataLoader(perm_dataset, batch_size=batch_size,
                                     shuffle=False,
                                     collate_fn=collate_mortgage_sequences)
            perm_cif = []
            with torch.no_grad():
                for x_pad, lens, _, _ in perm_loader:
                    cif = model.predict_cif(x_pad.to(device), lens.to(device))
                    perm_cif.append(cif[:, cause_idx, :].cpu().numpy())
            perm_cif = np.concatenate(perm_cif)
            perm_risk = perm_cif.mean(axis=1)
            perm_c = concordance_index_censored(event_binary, durations, perm_risk)[0]
            scores.append(baseline_c - perm_c)

        importances.append(np.mean(scores))
        importances_std.append(np.std(scores))

    return np.array(importances), np.array(importances_std)

# Prepay importance
print('Computing prepayment importance...')
imp_prepay, imp_prepay_std = permutation_importance_dynamic(
    model, test_seqs, final_feature_cols, cause_idx=0,
    device=DEVICE, time_points=time_points, max_seq_len=MAX_SEQ_LEN)

importance_prepay = pd.DataFrame({
    'feature': final_feature_cols,
    'importance': imp_prepay,
    'std': imp_prepay_std
}).sort_values('importance', ascending=False)

print('\nComputing default importance...')
imp_default, imp_default_std = permutation_importance_dynamic(
    model, test_seqs, final_feature_cols, cause_idx=1,
    device=DEVICE, time_points=time_points, max_seq_len=MAX_SEQ_LEN)

importance_default = pd.DataFrame({
    'feature': final_feature_cols,
    'importance': imp_default,
    'std': imp_default_std
}).sort_values('importance', ascending=False)

print('\nTop 10 Features - PREPAYMENT:')
print(importance_prepay.head(10).to_string(index=False))
print('\nTop 10 Features - DEFAULT:')
print(importance_default.head(10).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 8))

for ax, df, color, title in zip(axes,
    [importance_prepay, importance_default],
    ['steelblue', 'indianred'],
    ['Prepayment', 'Default'],
):
    top_n = 15
    plot_df = df.head(top_n).iloc[::-1]
    ax.barh(plot_df['feature'], plot_df['importance'],
            xerr=plot_df['std'], color=color, alpha=0.7, capsize=3)
    ax.set_xlabel('Importance (decrease in C-index)')
    ax.set_title(f'Dynamic-DeepHit Permutation Importance: {title}')
    ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'dynamic_deephit_feature_importance.png',
            dpi=150, bbox_inches='tight')
plt.show()

---

## Save Model Artifacts

In [ ]:
# Save model checkpoint
torch.save({
    'model_state_dict': model.state_dict(),
    'in_features': in_features,
    'num_time_bins': NUM_TIME_BINS,
    'num_causes': 2,
    'embed_dim': DDEEPHIT_PARAMS['embed_dim'],
    'hidden_dim': DDEEPHIT_PARAMS['hidden_dim'],
    'num_rnn_layers': DDEEPHIT_PARAMS['num_rnn_layers'],
    'head_hidden1': DDEEPHIT_PARAMS['head_hidden1'],
    'head_hidden2': DDEEPHIT_PARAMS['head_hidden2'],
    'dropout': DDEEPHIT_PARAMS['dropout'],
    'num_tv_features': len(tv_feature_indices),
    'tv_feature_indices': tv_feature_indices,
}, MODELS_DIR / 'dynamic_deephit_joint.pt')

# Save scaler
with open(MODELS_DIR / 'dynamic_deephit_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# Save time bins
np.save(MODELS_DIR / 'dynamic_deephit_time_bins.npy', time_bins)

# Save feature columns
with open(MODELS_DIR / 'dynamic_deephit_feature_cols.pkl', 'wb') as f:
    pickle.dump(final_feature_cols, f)

# Save training history
with open(MODELS_DIR / 'dynamic_deephit_history.pkl', 'wb') as f:
    pickle.dump(history, f)

# Save importance
importance_prepay.to_csv(MODELS_DIR / 'dynamic_deephit_importance_prepay.csv', index=False)
importance_default.to_csv(MODELS_DIR / 'dynamic_deephit_importance_default.csv', index=False)

print(f'All artifacts saved to {MODELS_DIR}:')
print(f'  - dynamic_deephit_joint.pt')
print(f'  - dynamic_deephit_scaler.pkl')
print(f'  - dynamic_deephit_time_bins.npy')
print(f'  - dynamic_deephit_feature_cols.pkl')
print(f'  - dynamic_deephit_history.pkl')
print(f'  - dynamic_deephit_importance_prepay.csv')
print(f'  - dynamic_deephit_importance_default.csv')

---

## Summary

In [ ]:
print('=' * 70)
print('DYNAMIC-DEEPHIT (Lee et al. 2020) - SUMMARY')
print('=' * 70)

print(f'\nArchitecture:')
print(f'  Input Embedding: {in_features} -> {DDEEPHIT_PARAMS["embed_dim"]}')
print(f'  GRU: {DDEEPHIT_PARAMS["num_rnn_layers"]} layers x {DDEEPHIT_PARAMS["hidden_dim"]} hidden')
print(f'  Temporal Attention over all hidden states')
print(f'  Cause heads: {DDEEPHIT_PARAMS["head_hidden1"]} -> {DDEEPHIT_PARAMS["head_hidden2"]} -> {NUM_TIME_BINS}')
print(f'  Output: Joint softmax over {2 * NUM_TIME_BINS} (cause, time)')
print(f'  Parameters: {n_params:,} (vs ~722K for static DeepHit)')

print(f'\nData:')
print(f'  Train: {len(train_seqs):,} sequences')
print(f'  Val: {len(val_seqs):,} sequences')
print(f'  Test: {len(test_seqs):,} sequences')
print(f'  Max seq length: {MAX_SEQ_LEN}')
print(f'  Features: {len(final_feature_cols)}')

print(f'\nLoss function: L1 (NLL) + L2 (Ranking) + L3 (Next-Step)')
print(f'  alpha_prepay={DDEEPHIT_PARAMS["alpha_prepay"]}, '
      f'alpha_default={DDEEPHIT_PARAMS["alpha_default"]}')
print(f'  beta={DDEEPHIT_PARAMS["beta"]}, sigma={DDEEPHIT_PARAMS["sigma"]}')
print(f'  default_event_weight={DDEEPHIT_PARAMS["default_event_weight"]}')

print(f'\nTraining:')
print(f'  Best epoch: {best_epoch}')
print(f'  Best val loss: {best_val_loss:.4f}')

print(f'\nKey advantage over static DeepHit:')
print(f'  - Uses full monthly history (not just terminal observation)')
print(f'  - Dynamic predictions that improve with more data')
print(f'  - Temporal attention reveals which months matter most')
print(f'  - L3 loss regularizes RNN via next-step prediction')

---

## Next Steps

- Compare Dynamic-DeepHit vs static DeepHit performance
- Investigate attention patterns for prepay vs default loans
- Ablation studies: L3 weight, sequence length, GRU depth
- Time-varying feature importance analysis